In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

In [ ]:
with open("SEPTUAGINT.xml") as f:
    sept_soup = BeautifulSoup(f.read(), 'xml')

with open("TISCHENDORF.xml") as file:
    tisch_soup = BeautifulSoup(file, 'xml')


rm_punct_tbl = str.maketrans("", "", " .·,:;!()«»-·")
sept = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str", "G").removeprefix("G"),
            e.find_parent("VERS")["vnumber"],
            e.find_parent("CHAPTER")["cnumber"],
            e.find_parent("BIBLEBOOK")["bnumber"],
            e["rmac"],
        )
        for e in sept_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)
tisch = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str"),
            e.find_parent("VERS")["vnumber"],
            e.find_parent("CHAPTER")["cnumber"],
            e.find_parent("BIBLEBOOK")["bnumber"],
            e["rmac"],
        )
        for e in tisch_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)

In [ ]:
sept["verse"] = sept["verse"].astype(int)
sept["chapter"] = sept["chapter"].astype(int)
sept["book"] = sept["book"].astype(int)
# include code to parse rmac?

In [ ]:
tisch["verse"] = tisch["verse"].astype(int)
tisch["chapter"] = tisch["chapter"].astype(int)
tisch["book"] = tisch["book"].astype(int)

## Reconciling missing strongs

In [ ]:
# words without strongs
nameless = pd.concat(
    [tisch[~tisch["str"].astype(bool)], sept[~sept["str"].astype(bool)]]
)

# unique
no_str_words = pd.Series(nameless["text"].unique())


new_strongs = pd.Series(
    ['C' + str(number) for number in range(0, len(no_str_words))],
    index=no_str_words
)

In [ ]:
# if null or "" sets to False, anything else -> True.
tisch_needs_new = ~tisch["str"].astype(bool)
tisch.loc[tisch_needs_new, "str"] = new_strongs.loc[
    tisch.loc[tisch_needs_new, "text"]
].values

sept_needs_new = ~sept["str"].astype(bool)
sept.loc[sept_needs_new, "str"] = new_strongs.loc[
    sept.loc[sept_needs_new, "text"]
].values

In [ ]:
import pickle
with open('./pickles/tisch.pickle', 'wb') as f:
    pickle.dump(tisch, f)
with open('./pickles/sept.pickle', 'wb') as f:
    pickle.dump(sept, f)